In [ ]:
import pandas as pd
import os

In [ ]:

# Paths to your crop CSV files
uploaded_files = {
    "wheat": r"C:\Users\ANKIT BHAR\Desktop\Ideas project\merged_wheat_reservoir (1).csv",
    "rabi_rice": r"C:\Users\ANKIT BHAR\Desktop\Ideas project\merged_rabi_rice_reservoir.csv",
    "potato": r"C:\Users\ANKIT BHAR\Desktop\Ideas project\merged_potato_reservoir.csv",
    "mustard": r"C:\Users\ANKIT BHAR\Desktop\Ideas project\merged_mustard_reservoir.csv",
    "massor": r"C:\Users\ANKIT BHAR\Desktop\Ideas project\merged_massor_reservoir.csv",
    "gram": r"C:\Users\ANKIT BHAR\Desktop\Ideas project\merged_gram_reservoir.csv"
}

In [ ]:
# Column mapping
column_mapping = {
    'temperature_recorded_date': 'DATE',
    'state_name': 'STATE',
    'state_temperature_max_val': 'TEMP_MAX',
    'state_temperature_min_val': 'TEMP_MIN',
    'state_rainfall_val': 'RAINFALL',
    'Level': 'WATER_LEVEL',
    'Current Live Storage': 'WATER_LIVE_STORAGE',
    'FRL' : 'FRL',
    'Live Cap FRL' : 'LIVE_CAP_FRL',
    'yield' : 'Yield'
}

# Create output folder
output_dir = "weekly_crop_data"
os.makedirs(output_dir, exist_ok=True)

# Process each crop file
for crop, path in uploaded_files.items():
    df = pd.read_csv(path, parse_dates=['temperature_recorded_date'])
    df = df.rename(columns=column_mapping)

    # Create YEAR and WEEK columns
    df['YEAR'] = df['DATE'].dt.year
    df['WEEK'] = df['DATE'].dt.isocalendar().week

    # Aggregate weekly data
    avg_cols = ['TEMP_MAX', 'TEMP_MIN', 'WATER_LEVEL', 'WATER_LIVE_STORAGE' , 'FRL' , 'LIVE_CAP_FRL' , 'Yield']
    sum_cols = ['RAINFALL']
    grouped = df.groupby(['STATE', 'YEAR', 'WEEK'])
    weekly = grouped[avg_cols].mean()
    weekly[sum_cols] = grouped[sum_cols].sum()
    weekly = weekly.reset_index()

    # Save state-wise weekly data
    for state in weekly['STATE'].unique():
        state_df = weekly[weekly['STATE'] == state]
        filename = f"{crop}_{state}_weekly.csv".replace(" ", "_").lower()
        state_df.to_csv(os.path.join(output_dir, filename), index=False, float_format='%.5f')


In [ ]:
# Column mapping
column_mapping = {
    'temperature_recorded_date': 'DATE',
    'state_name': 'STATE',
    'state_temperature_max_val': 'TEMP_MAX',
    'state_temperature_min_val': 'TEMP_MIN',
    'state_rainfall_val': 'RAINFALL',
    'Level': 'WATER_LEVEL',
    'Current Live Storage': 'WATER_LIVE_STORAGE',
    'FRL' : 'FRL',
    'Live Cap FRL' : 'LIVE_CAP_FRL',
    'yield': 'Yield'
}

# Create output folder
output_dir = "weekly_crop_data_extreme_temp"
os.makedirs(output_dir, exist_ok=True)

# Process each crop file
for crop, path in uploaded_files.items():
    df = pd.read_csv(path, parse_dates=['temperature_recorded_date'])
    df = df.rename(columns=column_mapping)

    # Add YEAR and WEEK columns
    df['YEAR'] = df['DATE'].dt.year
    df['WEEK'] = df['DATE'].dt.isocalendar().week

    # Grouping
    grouped = df.groupby(['STATE', 'YEAR', 'WEEK'])

    # Compute required weekly stats
    weekly = pd.DataFrame({
        'TEMP_MAX': grouped['TEMP_MAX'].mean(),  # max of TEMP_MAX
        'TEMP_MIN': grouped['TEMP_MIN'].mean(),  # min of TEMP_MIN
        'WATER_LEVEL': grouped['WATER_LEVEL'].mean(),
        'WATER_LIVE_STORAGE': grouped['WATER_LIVE_STORAGE'].mean(),
        'RAINFALL': grouped['RAINFALL'].sum(),
        'FRL' : grouped['FRL'].mean(),
        'LIVE_CAP_FRL' : grouped['LIVE_CAP_FRL'].mean(),
        'Yield': grouped['Yield'].mean()
    }).reset_index()

    # Save each state's weekly data
    for state in weekly['STATE'].unique():
        state_df = weekly[weekly['STATE'] == state]
        filename = f"{crop}_{state}_weekly_extremetemp.csv".replace(" ", "_").lower()
        state_df.to_csv(os.path.join(output_dir, filename), index=False ,  float_format='%.5f')

In [ ]:


# Input and output folder paths
input_folder = "weekly_crop_data"  # Replace with your folder path if different
output_folder = "last12weeks_output"
os.makedirs(output_folder, exist_ok=True)

# Loop over all CSV files in the folder
for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):
        file_path = os.path.join(input_folder, filename)

        # Read the file
        df = pd.read_csv(file_path)

        # Ensure data types
        df['WEEK'] = df['WEEK'].astype(int)
        df['YEAR'] = df['YEAR'].astype(int)

        # Only keep relevant columns
        keep_cols = ['YEAR', 'WEEK', 'TEMP_MAX', 'TEMP_MIN', 'RAINFALL', 'WATER_LIVE_STORAGE', 'Yield']
        df = df[keep_cols]

        # Get yearly average yield
        Yield = df.groupby('YEAR')['Yield'].mean().reset_index()

        # Filter last 12 weeks (weeks 41–52)
        df_last12 = df[df['WEEK'].between(41, 52)]

        # Pivot to wide format
        pivoted = df_last12.pivot(index='YEAR', columns='WEEK', values=['TEMP_MAX', 'TEMP_MIN', 'RAINFALL', 'WATER_LIVE_STORAGE'])
        pivoted.columns = [f"{col}_{week}" for col, week in pivoted.columns]
        pivoted = pivoted.reset_index()

        # Merge with average yield
        final_df = pd.merge(pivoted, Yield, on='YEAR')

        # Round to 5 decimals
        final_df = final_df.round(5)

        # Save output to new folder
        output_file = os.path.join(output_folder, f"last12weeks_{filename}")
        final_df.to_csv(output_file, index=False)


In [ ]:


# Input and output folder paths
input_folder = "weekly_crop_data_extreme_temp"  # Replace with your folder path if different
output_folder = "last12weeks_tempextreme_output"
os.makedirs(output_folder, exist_ok=True)

# Loop over all CSV files in the folder
for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):
        file_path = os.path.join(input_folder, filename)

        # Read the file
        df = pd.read_csv(file_path)

        # Ensure data types
        df['WEEK'] = df['WEEK'].astype(int)
        df['YEAR'] = df['YEAR'].astype(int)

        # Only keep relevant columns
        keep_cols = ['YEAR', 'WEEK', 'TEMP_MAX', 'TEMP_MIN', 'RAINFALL', 'WATER_LIVE_STORAGE', 'Yield']
        df = df[keep_cols]

        # Get yearly average yield
        yield_per_year = df.groupby('YEAR')['Yield'].mean().reset_index()

        # Filter last 12 weeks (weeks 41–52)
        df_last12 = df[df['WEEK'].between(41, 52)]

        # Pivot to wide format
        pivoted = df_last12.pivot(index='YEAR', columns='WEEK', values=['TEMP_MAX', 'TEMP_MIN', 'RAINFALL', 'WATER_LIVE_STORAGE'])
        pivoted.columns = [f"{col}_{week}" for col, week in pivoted.columns]
        pivoted = pivoted.reset_index()

        # Merge with average yield
        final_df = pd.merge(pivoted, yield_per_year, on='YEAR')

        # Round to 5 decimals
        final_df = final_df.round(5)

        # Save output to new folder
        output_file = os.path.join(output_folder, f"last12weeks_{filename}")
        final_df.to_csv(output_file, index=False)


In [ ]:


# Specify your folder here
input_folder = "last12weeks_output"  # Change this to your actual folder

# Track issues
issues = []

# Loop through all CSV files
for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):
        path = os.path.join(input_folder, filename)
        try:
            df = pd.read_csv(path)

            if 'YEAR' not in df.columns or 'Yield' not in df.columns:
                issues.append((filename, "Missing YEAR or Yield column"))
                continue

            # Drop rows with missing YEAR or Yield
            df = df[['YEAR', 'Yield']].dropna(subset=['YEAR'])

            # Convert YEAR to int
            df['YEAR'] = df['YEAR'].astype(int)

            # Check 1: Years with missing Yield
            missing_yield_years = df[df['Yield'].isna()]['YEAR'].unique().tolist()

            # Check 2: Total unique years
            unique_years = sorted(df['YEAR'].unique())
            num_years = len(unique_years)

            # Check 3: Missing years in between
            expected_years = set(range(min(unique_years), max(unique_years)+1))
            missing_years = sorted(expected_years - set(unique_years))

            # Report issues
            file_issues = []
            if missing_yield_years:
                file_issues.append(f"Missing Yield for years: {missing_yield_years}")
            if num_years < 10:
                file_issues.append(f"Only {num_years} years of data")
            if missing_years:
                file_issues.append(f"Missing years in sequence: {missing_years}")
            if file_issues:
                issues.append((filename, "; ".join(file_issues)))

        except Exception as e:
            issues.append((filename, f"Error processing file: {str(e)}"))

# Print results
if issues:
    print("Files with data issues:")
    for fname, problem in issues:
        print(f"- {fname}: {problem}")
else:
    print(" All files passed the checks.")


In [ ]:
import os
import pandas as pd

# Input and output settings
input_folder = "last12weeks_output"  # Replace with your actual folder path
output_report = "data_issues_summary.csv"

# List to store issue records
issues = []

# Loop through CSV files in the folder
for filename in os.listdir(input_folder):
    if filename.endswith(".csv"):
        path = os.path.join(input_folder, filename)
        try:
            df = pd.read_csv(path)

            if 'YEAR' not in df.columns or 'Yield' not in df.columns:
                issues.append({
                    "File": filename,
                    "Issue": "Missing YEAR or Yield column"
                })
                continue

            df = df[['YEAR', 'Yield']].dropna(subset=['YEAR'])
            df['YEAR'] = df['YEAR'].astype(int)

            # Check 1: Years with missing yield
            missing_yield_years = df[df['Yield'].isna()]['YEAR'].unique().tolist()

            # Check 2: Total years
            unique_years = sorted(df['YEAR'].unique())
            num_years = len(unique_years)

            # Check 3: Missing in sequence
            expected_years = set(range(min(unique_years), max(unique_years)+1))
            missing_sequence_years = sorted(expected_years - set(unique_years))

            # Collect issues
            if missing_yield_years:
                issues.append({
                    "File": filename,
                    "Issue": f"Missing Yield for years: {missing_yield_years}"
                })
            if num_years < 10:
                issues.append({
                    "File": filename,
                    "Issue": f"Only {num_years} years of data"
                })
            if missing_sequence_years:
                issues.append({
                    "File": filename,
                    "Issue": f"Missing years in sequence: {missing_sequence_years}"
                })

        except Exception as e:
            issues.append({
                "File": filename,
                "Issue": f"Error processing file: {str(e)}"
            })

# Create report as DataFrame
report_df = pd.DataFrame(issues)

# Save to CSV
report_df.to_csv(output_report, index=False)

print(f" Report saved as {output_report}")
